# 04 · Compare the four conditions

Run the same questions through base+documents, base+RAG, adapter+documents and adapter+RAG. The model, generation budget and prompt template are shared. Base/adapter prompts must match for the same evidence mode. No condition gets extra numeric tools.

**Historical 2025 sources. No fine-tuning or paid API runs automatically.** Every notebook includes the same sample so you can open it independently. For your own documents, transfer the exported ZIP artifacts between notebooks.

In [ ]:
#@title 1. Load the included workshop code and sample sources
import base64, hashlib, io, json, sys, tempfile, zipfile
from pathlib import Path
payload = base64.b64decode("...")
assert hashlib.sha256(payload).hexdigest() == "c52e11f513adc84c2da3318f89d6e277e404a6bb83974cdb512e8cc519721d4d"
ROOT = Path(tempfile.gettempdir()) / "aca-workshop-c52e11f513ad"
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for name in z.namelist():
        target = ROOT / name
        if not target.resolve().is_relative_to(ROOT.resolve()): raise ValueError("Unsafe package path")
        data = z.read(name)
        if target.exists():
            if target.read_bytes() != data: raise ValueError("Existing workshop code changed; use a fresh runtime")
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with target.open("xb") as f: f.write(data)
sys.path.insert(0, str(ROOT))
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)
config = json.loads((ROOT / "config/rag.json").read_text())
print("Workshop source snapshot loaded. Nothing has been trained or generated yet.")

In [ ]:
#@title 2. Install the pinned notebook dependencies
INSTALL_DEPENDENCIES = True #@param {type:"boolean"}
print((ROOT / "requirements-rag.txt").read_text())
if INSTALL_DEPENDENCIES:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-rag.txt")], check=True)
from src.rag.notebook import unique, upload_artifact, export_artifact, download
from src.rag.corpus import load_corpus
from src.rag.core import digest
from src.rag.ui import show_corpus, show_search, show_answer, show_summary, show_comparison
print("Dependencies ready. Use a GPU for extended generation/training; retrieval can run on CPU.")

In [ ]:
#@title Choose the corpus saved by notebook 01, or the identical included sample
CORPUS_SOURCE = "Sample" #@param ["Sample", "Upload corpus artifact"]
CORPUS = ROOT / "sample" if CORPUS_SOURCE == "Sample" else upload_artifact("corpus", OUTPUTS)
docs, manifest = load_corpus(CORPUS)
show_corpus(docs, manifest)

In [ ]:
#@title Build or import the BM25 + semantic + graph index
INDEX_SOURCE = "Build" #@param ["Build", "Upload index artifact"]
from src.rag.semantic import Embedder, build_index
from src.rag.hybrid import Hybrid
embedder = Embedder(config)
if INDEX_SOURCE == "Build":
    INDEX = build_index(CORPUS, unique(OUTPUTS, "index"), embedder)
else:
    INDEX = upload_artifact("index", OUTPUTS)
rag = Hybrid(CORPUS, INDEX, embedder)
print("Index verified against this exact corpus and embedding configuration.")

### Load the adapter and choose validation or final test
Use validation while changing retrieval or training settings. Switch to test only after freezing choices. Without an adapter, the two adapter conditions are explicitly marked unrun.

In [ ]:
USE_ADAPTER = False #@param {type:"boolean"}
ADAPTER = upload_artifact("adapter", OUTPUTS) if USE_ADAPTER else None
SPLIT = "validation" #@param ["validation", "test"]
QUESTIONS_PER_DATASET = 2 #@param {type:"integer"}
from src.rag.experiment import training_rows
from src.rag.evaluate import questions_from_rows
if ADAPTER:
    dataset = json.loads((ADAPTER / "dataset.json").read_text())
    rows = dataset[SPLIT]
    print("Adapter smoke-only:", json.loads((ADAPTER / "training.json").read_text())["smoke_only"])
elif manifest["sources"].get("data/raw/state.csv"):
    _, validation_rows, test_rows = training_rows(ROOT)
    rows = validation_rows if SPLIT == "validation" else test_rows
else:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError("Upload one evaluation JSONL file")
    rows = [json.loads(line) for line in next(iter(uploaded.values())).decode().splitlines() if line.strip()]
if rows and "namespace" in rows[0] and "filters" not in rows[0]:
    rows = questions_from_rows(rows)
questions = []
for ns in ["enrollment", "plans"]:
    questions += [r for r in rows if r.get("namespace", r.get("filters",{}).get("namespace")) == ns][:QUESTIONS_PER_DATASET]
if not questions: raise ValueError("Evaluation questions need namespace or filters.namespace")
print("Selected", len(questions), SPLIT, "questions. No reference answers are passed to the models.")

### Check retrieval separately
These source-recall scores use existing citation sets as partial judgments. They are useful diagnostics, not a complete measure of relevance or answer accuracy.

In [ ]:
RUN_RETRIEVAL_EVAL = True #@param {type:"boolean"}
if RUN_RETRIEVAL_EVAL:
    from src.rag.evaluate import retrieval_ablation
    retrieval_report = retrieval_ablation(rag, questions, unique(OUTPUTS, "retrieval-eval"))
    for group in retrieval_report["summary"]: print(group)

### Run the paired model comparison
**Strict** document mode can fail when all selected text exceeds the budget; those failures remain visible. **Prefix** is an explicit partial-document baseline with a reported omission count. Neither mode does query-based document ranking. Adapter conditions use exactly the same document or RAG prompt as their base counterparts.

In [ ]:
RUN_COMPARISON = False #@param {type:"boolean"}
DEVICE = "auto" #@param ["auto", "cpu", "cuda"]
DOCUMENT_POLICY = "prefix" #@param ["prefix", "strict"]
if RUN_COMPARISON:
    from src.rag.experiment import comparison
    RUN = comparison(questions, CORPUS, INDEX, embedder, config, unique(OUTPUTS, "comparison"),
                     adapter=ADAPTER, device=DEVICE, policy=DOCUMENT_POLICY)
    show_summary(json.loads((RUN / "summary.json").read_text()))
    comparison_rows = [json.loads(line) for line in (RUN / "predictions.jsonl").read_text().splitlines()]
    show_comparison(comparison_rows)
    results_zip = export_artifact(RUN, "comparison", OUTPUTS)
else:
    print("Comparison not run. Enable RUN_COMPARISON when the selected artifacts and split are ready.")

### Review the answers before declaring a winner
The results artifact contains exact prompts, retrieved passages, graph paths, timings, error rows, a shuffled manual-review worksheet and a separate condition key. Grade correctness, citation support and preserved caveats. Review enrollment and plan benefits separately.

In [ ]:
DOWNLOAD_RESULTS = False #@param {type:"boolean"}
if DOWNLOAD_RESULTS:
    if not RUN_COMPARISON: raise ValueError("Run the comparison first")
    download(results_zip)